# DBSCAN - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import DBSCAN as SklearnDBSCAN
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is DBSCAN?

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** is a density-based clustering algorithm that groups together points that are closely packed (points with many nearby neighbors), marking points in low-density regions as outliers.

### Key Concepts

#### Core Points, Border Points, and Noise Points

DBSCAN classifies each point into one of three categories:

1. **Core Point**: A point that has at least `min_samples` points within distance `eps` (including itself)
   - Forms the "dense" regions of clusters
   
2. **Border Point**: A point that is within `eps` distance of a core point but doesn't have `min_samples` neighbors itself
   - On the edge of clusters
   
3. **Noise Point (Outlier)**: A point that is neither a core point nor a border point
   - Labeled as -1 in the output

### Parameters

#### eps (epsilon)
- The maximum distance between two points for them to be considered neighbors
- Defines the radius of the neighborhood around each point
- Too small: Many points become noise
- Too large: Clusters merge together

#### min_samples
- The minimum number of points required to form a dense region (core point)
- Typically set to: `min_samples >= D + 1` where D is the dimensionality
- Higher values: More robust to noise, but may miss small clusters
- Lower values: More sensitive to noise

### Density Reachability and Connectivity

#### Directly Density-Reachable
Point $p$ is **directly density-reachable** from point $q$ if:
- $q$ is a core point
- $p$ is within the $\epsilon$-neighborhood of $q$: $||p - q|| \leq \epsilon$

#### Density-Reachable
Point $p$ is **density-reachable** from point $q$ if there is a chain of points $p_1, p_2, ..., p_n$ where:
- $p_1 = q$ and $p_n = p$
- Each $p_{i+1}$ is directly density-reachable from $p_i$

#### Density-Connected
Points $p$ and $q$ are **density-connected** if there exists a point $o$ such that both $p$ and $q$ are density-reachable from $o$.

### Algorithm Steps

1. For each unvisited point $p$:
   - Mark $p$ as visited
   - Find all neighbors within distance `eps`
   - If `|neighbors| >= min_samples`:
     - $p$ is a core point, start a new cluster
     - Expand cluster by adding all density-reachable points
   - Else: Mark $p$ as noise (may later become a border point)

### Time and Space Complexity

| Aspect | Without Spatial Index | With Spatial Index (KD-Tree/Ball-Tree) |
|--------|----------------------|----------------------------------------|
| Time | $O(n^2)$ | $O(n \log n)$ average case |
| Space | $O(n)$ | $O(n)$ |

**Note**: The $O(n \log n)$ complexity with spatial indexing assumes low-dimensional data. For high dimensions, it degrades to $O(n^2)$.

### Advantages of DBSCAN
- Does not require specifying the number of clusters (K)
- Can find arbitrarily shaped clusters
- Robust to outliers (identifies them explicitly)
- Only two parameters to tune

### Limitations
- Struggles with clusters of varying densities
- Sensitive to the choice of eps and min_samples
- Performance degrades in high dimensions (curse of dimensionality)

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class DBSCANScratch:
    """
    DBSCAN (Density-Based Spatial Clustering of Applications with Noise) 
    implementation from scratch.
    
    Parameters:
    -----------
    eps : float, default=0.5
        The maximum distance between two samples to be considered neighbors.
    min_samples : int, default=5
        The minimum number of samples in a neighborhood to form a core point.
    metric : str, default='euclidean'
        Distance metric to use ('euclidean' or 'manhattan').
    
    Attributes:
    -----------
    labels_ : ndarray of shape (n_samples,)
        Cluster labels for each point. Noisy samples are given label -1.
    core_sample_indices_ : ndarray
        Indices of core samples.
    n_clusters_ : int
        Number of clusters found (excluding noise).
    n_noise_ : int
        Number of noise points.
    """
    
    def __init__(self, eps=0.5, min_samples=5, metric='euclidean'):
        self.eps = eps
        self.min_samples = min_samples
        self.metric = metric
        self.labels_ = None
        self.core_sample_indices_ = None
        self.n_clusters_ = 0
        self.n_noise_ = 0
        
    def _compute_distance(self, x1, x2):
        """
        Compute distance between two points based on the chosen metric.
        
        Parameters:
        -----------
        x1 : ndarray
            First point.
        x2 : ndarray
            Second point.
            
        Returns:
        --------
        float
            Distance between x1 and x2.
        """
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        elif self.metric == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        else:
            raise ValueError(f"Unknown metric: {self.metric}")
    
    def _compute_distance_matrix(self, X):
        """
        Compute pairwise distance matrix for all points.
        Optimized using vectorization for efficiency.
        
        Parameters:
        -----------
        X : ndarray of shape (n_samples, n_features)
            Input data.
            
        Returns:
        --------
        ndarray of shape (n_samples, n_samples)
            Pairwise distance matrix.
        """
        n_samples = X.shape[0]
        
        if self.metric == 'euclidean':
            # Efficient vectorized computation using (a-b)^2 = a^2 + b^2 - 2ab
            sq_norms = np.sum(X ** 2, axis=1)
            distances = sq_norms[:, np.newaxis] + sq_norms[np.newaxis, :] - 2 * np.dot(X, X.T)
            # Handle numerical errors (negative values close to zero)
            distances = np.maximum(distances, 0)
            distances = np.sqrt(distances)
        elif self.metric == 'manhattan':
            # Manhattan distance requires different approach
            distances = np.zeros((n_samples, n_samples))
            for i in range(n_samples):
                distances[i, :] = np.sum(np.abs(X - X[i]), axis=1)
        else:
            raise ValueError(f"Unknown metric: {self.metric}")
            
        return distances
    
    def _get_neighbors(self, distance_matrix, point_idx):
        """
        Find all neighbors of a point within eps distance.
        
        Parameters:
        -----------
        distance_matrix : ndarray
            Precomputed distance matrix.
        point_idx : int
            Index of the point.
            
        Returns:
        --------
        ndarray
            Indices of neighboring points.
        """
        return np.where(distance_matrix[point_idx] <= self.eps)[0]
    
    def _expand_cluster(self, distance_matrix, labels, point_idx, neighbors, cluster_id, is_core):
        """
        Expand cluster from a core point by finding all density-reachable points.
        
        Parameters:
        -----------
        distance_matrix : ndarray
            Precomputed distance matrix.
        labels : ndarray
            Current cluster labels (-1 for unvisited/noise).
        point_idx : int
            Index of the starting core point.
        neighbors : ndarray
            Indices of neighbors of the starting point.
        cluster_id : int
            Current cluster ID to assign.
        is_core : ndarray
            Boolean array indicating which points are core points.
        """
        # Assign cluster to the starting point
        labels[point_idx] = cluster_id
        
        # Use a list as a queue for BFS-style expansion
        seed_set = list(neighbors)
        i = 0
        
        while i < len(seed_set):
            current_point = seed_set[i]
            
            # If this point was previously marked as noise, it becomes a border point
            if labels[current_point] == -1:
                labels[current_point] = cluster_id
            
            # If this point hasn't been assigned to any cluster yet
            elif labels[current_point] == -2:  # -2 means unvisited
                labels[current_point] = cluster_id
                
                # If this point is also a core point, add its neighbors to expand
                if is_core[current_point]:
                    current_neighbors = self._get_neighbors(distance_matrix, current_point)
                    for neighbor in current_neighbors:
                        if neighbor not in seed_set:
                            seed_set.append(neighbor)
            
            i += 1
    
    def fit(self, X):
        """
        Perform DBSCAN clustering on the input data.
        
        Parameters:
        -----------
        X : ndarray of shape (n_samples, n_features)
            Input data to cluster.
            
        Returns:
        --------
        self
            Fitted estimator.
        """
        X = np.array(X)
        n_samples = X.shape[0]
        
        # Compute distance matrix once (trade memory for speed)
        distance_matrix = self._compute_distance_matrix(X)
        
        # Initialize labels: -2 means unvisited
        labels = np.full(n_samples, -2, dtype=int)
        
        # Identify core points (vectorized)
        neighbor_counts = np.sum(distance_matrix <= self.eps, axis=1)
        is_core = neighbor_counts >= self.min_samples
        self.core_sample_indices_ = np.where(is_core)[0]
        
        cluster_id = 0
        
        # Process each point
        for point_idx in range(n_samples):
            # Skip if already processed
            if labels[point_idx] != -2:
                continue
            
            # Get neighbors
            neighbors = self._get_neighbors(distance_matrix, point_idx)
            
            # Check if core point
            if len(neighbors) < self.min_samples:
                # Mark as noise (may become border point later)
                labels[point_idx] = -1
            else:
                # Start new cluster from this core point
                self._expand_cluster(distance_matrix, labels, point_idx, 
                                    neighbors, cluster_id, is_core)
                cluster_id += 1
        
        self.labels_ = labels
        self.n_clusters_ = cluster_id
        self.n_noise_ = np.sum(labels == -1)
        
        return self
    
    def fit_predict(self, X):
        """
        Perform DBSCAN clustering and return cluster labels.
        
        Parameters:
        -----------
        X : ndarray of shape (n_samples, n_features)
            Input data to cluster.
            
        Returns:
        --------
        ndarray of shape (n_samples,)
            Cluster labels. Noise points are labeled as -1.
        """
        self.fit(X)
        return self.labels_
    
    def get_cluster_info(self):
        """
        Get summary information about the clustering results.
        
        Returns:
        --------
        dict
            Dictionary containing clustering summary.
        """
        if self.labels_ is None:
            raise ValueError("Model has not been fitted yet.")
            
        info = {
            'n_clusters': self.n_clusters_,
            'n_noise': self.n_noise_,
            'n_core_points': len(self.core_sample_indices_),
            'cluster_sizes': {}
        }
        
        for cluster_id in range(self.n_clusters_):
            info['cluster_sizes'][cluster_id] = np.sum(self.labels_ == cluster_id)
            
        return info

In [ ]:
# Quick test of the implementation
X_test = np.array([[1, 2], [2, 2], [2, 3], 
                   [8, 7], [8, 8], [9, 8],
                   [25, 80]])  # Noise point

dbscan_test = DBSCANScratch(eps=1.5, min_samples=2)
labels_test = dbscan_test.fit_predict(X_test)

print("Test clustering results:")
print(f"Labels: {labels_test}")
print(f"Number of clusters: {dbscan_test.n_clusters_}")
print(f"Number of noise points: {dbscan_test.n_noise_}")
print(f"Core sample indices: {dbscan_test.core_sample_indices_}")

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Generate different datasets for testing

# Dataset 1: Moons (non-convex shapes)
X_moons, y_moons = make_moons(n_samples=500, noise=0.1, random_state=42)

# Dataset 2: Circles (concentric circles)
X_circles, y_circles = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)

# Dataset 3: Custom data with noise
# Create blobs with different densities and add noise points
X_blobs, y_blobs = make_blobs(n_samples=400, centers=3, cluster_std=[0.5, 0.8, 1.2], 
                               random_state=42)
# Add noise points
noise_points = np.random.uniform(low=-10, high=10, size=(50, 2))
X_custom = np.vstack([X_blobs, noise_points])
y_custom = np.hstack([y_blobs, np.full(50, -1)])  # -1 for noise

# Standardize the data
scaler = StandardScaler()
X_moons_scaled = scaler.fit_transform(X_moons)
X_circles_scaled = scaler.fit_transform(X_circles)
X_custom_scaled = scaler.fit_transform(X_custom)

print(f"Moons dataset shape: {X_moons_scaled.shape}")
print(f"Circles dataset shape: {X_circles_scaled.shape}")
print(f"Custom dataset shape: {X_custom_scaled.shape}")

In [ ]:
# Visualize the datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=y_moons, cmap='viridis', s=20)
axes[0].set_title('Moons Dataset (True Labels)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_circles_scaled[:, 0], X_circles_scaled[:, 1], c=y_circles, cmap='viridis', s=20)
axes[1].set_title('Circles Dataset (True Labels)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

scatter = axes[2].scatter(X_custom_scaled[:, 0], X_custom_scaled[:, 1], c=y_custom, cmap='viridis', s=20)
axes[2].set_title('Custom Dataset with Noise (True Labels)')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
def find_optimal_eps(X, k=None, plot=True):
    """
    Use the k-distance plot (elbow method) to find optimal eps.
    
    Parameters:
    -----------
    X : ndarray
        Input data.
    k : int, optional
        Number of nearest neighbors. Default is min_samples - 1.
    plot : bool
        Whether to plot the k-distance graph.
        
    Returns:
    --------
    ndarray
        Sorted k-distances.
    """
    if k is None:
        k = min(5, X.shape[0] - 1)  # Default to 5 or less
    
    # Compute k-nearest neighbors
    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    distances, _ = nbrs.kneighbors(X)
    
    # Get the distance to the k-th nearest neighbor
    k_distances = distances[:, k-1]
    
    # Sort distances
    k_distances_sorted = np.sort(k_distances)
    
    if plot:
        plt.figure(figsize=(10, 5))
        plt.plot(range(len(k_distances_sorted)), k_distances_sorted, 'b-', linewidth=1)
        plt.xlabel('Points (sorted by distance)')
        plt.ylabel(f'{k}-Distance')
        plt.title(f'K-Distance Plot (k={k}) for Optimal eps Selection')
        plt.grid(True, alpha=0.3)
        
        # Find the "elbow" point using simple heuristic
        # Compute the distance from each point to the line connecting first and last points
        n_points = len(k_distances_sorted)
        all_coords = np.vstack([np.arange(n_points), k_distances_sorted]).T
        first_point = all_coords[0]
        last_point = all_coords[-1]
        line_vec = last_point - first_point
        line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
        vec_from_first = all_coords - first_point
        scalar_proj = np.dot(vec_from_first, line_vec_norm)
        proj_on_line = np.outer(scalar_proj, line_vec_norm) + first_point
        dist_to_line = np.sqrt(np.sum((all_coords - proj_on_line)**2, axis=1))
        elbow_idx = np.argmax(dist_to_line)
        
        elbow_eps = k_distances_sorted[elbow_idx]
        plt.axhline(y=elbow_eps, color='r', linestyle='--', 
                   label=f'Suggested eps = {elbow_eps:.3f}')
        plt.scatter([elbow_idx], [elbow_eps], color='r', s=100, zorder=5)
        plt.legend()
        plt.show()
        
        print(f"Suggested eps value (elbow point): {elbow_eps:.3f}")
    
    return k_distances_sorted

# Find optimal eps for each dataset
print("K-Distance plot for Moons dataset:")
_ = find_optimal_eps(X_moons_scaled, k=5)

In [ ]:
print("K-Distance plot for Circles dataset:")
_ = find_optimal_eps(X_circles_scaled, k=5)

In [ ]:
print("K-Distance plot for Custom dataset:")
_ = find_optimal_eps(X_custom_scaled, k=5)

In [ ]:
# Train DBSCAN on each dataset with tuned parameters

# Moons dataset
dbscan_moons = DBSCANScratch(eps=0.3, min_samples=5)
labels_moons = dbscan_moons.fit_predict(X_moons_scaled)
print("Moons Dataset:")
print(f"  Clusters found: {dbscan_moons.n_clusters_}")
print(f"  Noise points: {dbscan_moons.n_noise_}")
print(f"  Core points: {len(dbscan_moons.core_sample_indices_)}")

# Circles dataset
dbscan_circles = DBSCANScratch(eps=0.2, min_samples=5)
labels_circles = dbscan_circles.fit_predict(X_circles_scaled)
print("\nCircles Dataset:")
print(f"  Clusters found: {dbscan_circles.n_clusters_}")
print(f"  Noise points: {dbscan_circles.n_noise_}")
print(f"  Core points: {len(dbscan_circles.core_sample_indices_)}")

# Custom dataset
dbscan_custom = DBSCANScratch(eps=0.4, min_samples=5)
labels_custom = dbscan_custom.fit_predict(X_custom_scaled)
print("\nCustom Dataset:")
print(f"  Clusters found: {dbscan_custom.n_clusters_}")
print(f"  Noise points: {dbscan_custom.n_noise_}")
print(f"  Core points: {len(dbscan_custom.core_sample_indices_)}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
def evaluate_dbscan(X, labels, true_labels=None):
    """
    Evaluate DBSCAN clustering results with multiple metrics.
    
    Parameters:
    -----------
    X : ndarray
        Input data.
    labels : ndarray
        Cluster labels from DBSCAN.
    true_labels : ndarray, optional
        Ground truth labels for comparison.
        
    Returns:
    --------
    dict
        Dictionary containing evaluation metrics.
    """
    results = {
        'n_clusters': len(set(labels)) - (1 if -1 in labels else 0),
        'n_noise': np.sum(labels == -1),
        'noise_ratio': np.sum(labels == -1) / len(labels)
    }
    
    # Calculate silhouette score (excluding noise points)
    mask = labels != -1
    if len(set(labels[mask])) > 1:  # Need at least 2 clusters for silhouette
        results['silhouette_score'] = silhouette_score(X[mask], labels[mask])
    else:
        results['silhouette_score'] = None
    
    # Calculate Adjusted Rand Index if true labels provided
    if true_labels is not None:
        results['adjusted_rand_index'] = adjusted_rand_score(true_labels, labels)
    
    # Cluster size statistics
    cluster_sizes = []
    for cluster_id in range(results['n_clusters']):
        cluster_sizes.append(np.sum(labels == cluster_id))
    
    if cluster_sizes:
        results['cluster_sizes'] = cluster_sizes
        results['mean_cluster_size'] = np.mean(cluster_sizes)
        results['std_cluster_size'] = np.std(cluster_sizes)
        results['min_cluster_size'] = np.min(cluster_sizes)
        results['max_cluster_size'] = np.max(cluster_sizes)
    
    return results

# Evaluate all datasets
print("Moons Dataset Evaluation:")
print("-" * 40)
results_moons = evaluate_dbscan(X_moons_scaled, labels_moons, y_moons)
for key, value in results_moons.items():
    if key != 'cluster_sizes':
        print(f"  {key}: {value}")

print("\nCircles Dataset Evaluation:")
print("-" * 40)
results_circles = evaluate_dbscan(X_circles_scaled, labels_circles, y_circles)
for key, value in results_circles.items():
    if key != 'cluster_sizes':
        print(f"  {key}: {value}")

print("\nCustom Dataset Evaluation:")
print("-" * 40)
# For custom dataset, exclude the artificial noise labels from true_labels
results_custom = evaluate_dbscan(X_custom_scaled, labels_custom)
for key, value in results_custom.items():
    if key != 'cluster_sizes':
        print(f"  {key}: {value}")

In [ ]:
def plot_cluster_distribution(labels, title="Cluster Distribution"):
    """
    Plot the distribution of points across clusters.
    """
    unique_labels = np.unique(labels)
    counts = [np.sum(labels == label) for label in unique_labels]
    label_names = [f'Cluster {l}' if l != -1 else 'Noise' for l in unique_labels]
    
    colors = ['red' if l == -1 else plt.cm.viridis(l / max(1, max(unique_labels))) 
              for l in unique_labels]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Bar chart
    bars = axes[0].bar(label_names, counts, color=colors, edgecolor='black')
    axes[0].set_xlabel('Cluster')
    axes[0].set_ylabel('Number of Points')
    axes[0].set_title(f'{title} - Bar Chart')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, count in zip(bars, counts):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                    str(count), ha='center', va='bottom', fontsize=9)
    
    # Pie chart
    axes[1].pie(counts, labels=label_names, colors=colors, autopct='%1.1f%%',
               startangle=90, explode=[0.05 if l == -1 else 0 for l in unique_labels])
    axes[1].set_title(f'{title} - Pie Chart')
    
    plt.tight_layout()
    plt.show()

# Plot cluster distributions
plot_cluster_distribution(labels_moons, "Moons Dataset")
plot_cluster_distribution(labels_circles, "Circles Dataset")
plot_cluster_distribution(labels_custom, "Custom Dataset")

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
def plot_dbscan_results(X, labels, core_indices, title="DBSCAN Clustering Results"):
    """
    Visualize DBSCAN clustering results with noise highlighted.
    
    Parameters:
    -----------
    X : ndarray
        Input data.
    labels : ndarray
        Cluster labels.
    core_indices : ndarray
        Indices of core points.
    title : str
        Plot title.
    """
    unique_labels = set(labels)
    n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
    
    # Create color map
    colors = plt.cm.viridis(np.linspace(0, 1, max(n_clusters, 1)))
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create mask for core points
    is_core = np.zeros(len(X), dtype=bool)
    is_core[core_indices] = True
    
    for label in unique_labels:
        if label == -1:
            # Noise points
            mask = labels == label
            ax.scatter(X[mask, 0], X[mask, 1], c='red', marker='x', 
                      s=50, label='Noise', alpha=0.8, linewidth=2)
        else:
            mask = labels == label
            color = colors[label % len(colors)]
            
            # Border points (in cluster but not core)
            border_mask = mask & ~is_core
            ax.scatter(X[border_mask, 0], X[border_mask, 1], c=[color], 
                      marker='o', s=30, alpha=0.5, edgecolors='none')
            
            # Core points
            core_mask = mask & is_core
            ax.scatter(X[core_mask, 0], X[core_mask, 1], c=[color], 
                      marker='o', s=80, label=f'Cluster {label}', 
                      alpha=0.8, edgecolors='black', linewidth=1)
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(f'{title}\n(Large circles = Core points, Small circles = Border points, X = Noise)')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

# Visualize results for all datasets
plot_dbscan_results(X_moons_scaled, labels_moons, dbscan_moons.core_sample_indices_,
                   "DBSCAN on Moons Dataset")

In [ ]:
plot_dbscan_results(X_circles_scaled, labels_circles, dbscan_circles.core_sample_indices_,
                   "DBSCAN on Circles Dataset")

In [ ]:
plot_dbscan_results(X_custom_scaled, labels_custom, dbscan_custom.core_sample_indices_,
                   "DBSCAN on Custom Dataset")

In [ ]:
def plot_eps_effect(X, min_samples=5, eps_values=None):
    """
    Visualize the effect of different eps values on clustering.
    
    Parameters:
    -----------
    X : ndarray
        Input data.
    min_samples : int
        Fixed min_samples value.
    eps_values : list
        List of eps values to try.
    """
    if eps_values is None:
        eps_values = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
    
    n_plots = len(eps_values)
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    
    for idx, eps in enumerate(eps_values):
        dbscan = DBSCANScratch(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X)
        
        # Color points
        unique_labels = set(labels)
        colors = [plt.cm.viridis(l / max(1, max(unique_labels) if max(unique_labels) > 0 else 1)) 
                  if l != -1 else 'red' for l in labels]
        
        axes[idx].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.7)
        n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        axes[idx].set_title(f'eps={eps}\nClusters: {n_clusters}, Noise: {n_noise}')
        axes[idx].set_xlabel('Feature 1')
        axes[idx].set_ylabel('Feature 2')
    
    # Hide empty subplots
    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'Effect of eps Parameter (min_samples={min_samples})', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Show effect of eps on moons dataset
plot_eps_effect(X_moons_scaled, min_samples=5, eps_values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.8])

In [ ]:
def plot_min_samples_effect(X, eps=0.3, min_samples_values=None):
    """
    Visualize the effect of different min_samples values on clustering.
    
    Parameters:
    -----------
    X : ndarray
        Input data.
    eps : float
        Fixed eps value.
    min_samples_values : list
        List of min_samples values to try.
    """
    if min_samples_values is None:
        min_samples_values = [2, 3, 5, 10, 15, 20]
    
    n_plots = len(min_samples_values)
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    
    for idx, min_samples in enumerate(min_samples_values):
        dbscan = DBSCANScratch(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X)
        
        # Color points
        unique_labels = set(labels)
        colors = [plt.cm.viridis(l / max(1, max(unique_labels) if max(unique_labels) > 0 else 1)) 
                  if l != -1 else 'red' for l in labels]
        
        axes[idx].scatter(X[:, 0], X[:, 1], c=colors, s=20, alpha=0.7)
        n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        axes[idx].set_title(f'min_samples={min_samples}\nClusters: {n_clusters}, Noise: {n_noise}')
        axes[idx].set_xlabel('Feature 1')
        axes[idx].set_ylabel('Feature 2')
    
    # Hide empty subplots
    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'Effect of min_samples Parameter (eps={eps})', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Show effect of min_samples on moons dataset
plot_min_samples_effect(X_moons_scaled, eps=0.3, min_samples_values=[2, 3, 5, 10, 15, 20])

In [ ]:
def compare_dbscan_kmeans(X, dbscan_eps, dbscan_min_samples, kmeans_k, title=""):
    """
    Compare DBSCAN with K-Means on non-spherical data.
    
    Parameters:
    -----------
    X : ndarray
        Input data.
    dbscan_eps : float
        DBSCAN eps parameter.
    dbscan_min_samples : int
        DBSCAN min_samples parameter.
    kmeans_k : int
        Number of clusters for K-Means.
    title : str
        Title prefix for plots.
    """
    # Run DBSCAN
    dbscan = DBSCANScratch(eps=dbscan_eps, min_samples=dbscan_min_samples)
    dbscan_labels = dbscan.fit_predict(X)
    
    # Run K-Means
    kmeans = KMeans(n_clusters=kmeans_k, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(X)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # DBSCAN results
    unique_labels = set(dbscan_labels)
    colors_dbscan = [plt.cm.viridis(l / max(1, max(unique_labels) if max(unique_labels) > 0 else 1)) 
                     if l != -1 else 'red' for l in dbscan_labels]
    axes[0].scatter(X[:, 0], X[:, 1], c=colors_dbscan, s=30, alpha=0.7)
    n_clusters_dbscan = len(unique_labels) - (1 if -1 in dbscan_labels else 0)
    n_noise = np.sum(dbscan_labels == -1)
    axes[0].set_title(f'DBSCAN (eps={dbscan_eps}, min_samples={dbscan_min_samples})\n'
                     f'Clusters: {n_clusters_dbscan}, Noise: {n_noise}')
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    
    # K-Means results
    axes[1].scatter(X[:, 0], X[:, 1], c=kmeans_labels, cmap='viridis', s=30, alpha=0.7)
    axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                   c='red', marker='X', s=200, edgecolors='black', linewidth=2,
                   label='Centroids')
    axes[1].set_title(f'K-Means (K={kmeans_k})\nClusters: {kmeans_k}')
    axes[1].set_xlabel('Feature 1')
    axes[1].set_ylabel('Feature 2')
    axes[1].legend()
    
    plt.suptitle(f'{title} - DBSCAN vs K-Means Comparison', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Print silhouette scores
    mask_dbscan = dbscan_labels != -1
    if len(set(dbscan_labels[mask_dbscan])) > 1:
        sil_dbscan = silhouette_score(X[mask_dbscan], dbscan_labels[mask_dbscan])
        print(f"DBSCAN Silhouette Score (excluding noise): {sil_dbscan:.4f}")
    
    sil_kmeans = silhouette_score(X, kmeans_labels)
    print(f"K-Means Silhouette Score: {sil_kmeans:.4f}")

# Compare on moons dataset
print("Comparison on Moons Dataset:")
compare_dbscan_kmeans(X_moons_scaled, dbscan_eps=0.3, dbscan_min_samples=5, 
                     kmeans_k=2, title="Moons Dataset")

In [ ]:
# Compare on circles dataset
print("\nComparison on Circles Dataset:")
compare_dbscan_kmeans(X_circles_scaled, dbscan_eps=0.2, dbscan_min_samples=5, 
                     kmeans_k=2, title="Circles Dataset")

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use DBSCAN

#### Good Use Cases:

1. **Arbitrary-Shaped Clusters**
   - Data with non-spherical cluster shapes (crescents, rings, irregular shapes)
   - Spatial data (geographic clusters)
   - Image segmentation where objects have complex boundaries

2. **Noise/Outlier Detection**
   - Fraud detection in financial transactions
   - Anomaly detection in sensor data
   - Identifying defective products in manufacturing

3. **Unknown Number of Clusters**
   - Exploratory data analysis
   - Customer segmentation when segment count is unknown
   - Document clustering

4. **Spatial Applications**
   - Geographic hotspot detection (crime, disease outbreaks)
   - Traffic pattern analysis
   - Point of interest clustering

### When NOT to Use DBSCAN

#### Poor Use Cases:

1. **Varying Density Clusters**
   - When clusters have significantly different densities
   - Single eps cannot capture all clusters properly
   - Consider: OPTICS, HDBSCAN

2. **High-Dimensional Data**
   - Distance metrics become less meaningful (curse of dimensionality)
   - All points become equidistant
   - Consider: Dimensionality reduction first, or use subspace clustering

3. **Large Datasets Without Spatial Indexing**
   - O(n^2) complexity without index structures
   - Consider: Using sklearn with leaf_size optimization, or approximate methods

4. **Clusters Connected by Thin Bridges**
   - May incorrectly merge separate clusters
   - Consider: Spectral clustering

### Parameter Selection Guidelines

#### Selecting eps:

1. **K-Distance Plot Method** (Recommended)
   - Plot sorted k-distances for all points
   - Look for the "elbow" point
   - k should be set to min_samples - 1

2. **Domain Knowledge**
   - Use understanding of the data scale
   - Consider meaningful distance in the application context

3. **Grid Search**
   - Try multiple values and evaluate using silhouette score
   - Computationally expensive but thorough

#### Selecting min_samples:

1. **Rule of Thumb**: `min_samples >= D + 1` where D is dimensionality
   - For 2D data: min_samples >= 3
   - For higher dimensions: increase proportionally

2. **Dataset Size Consideration**:
   - Larger datasets: Use larger min_samples
   - Common range: 3-10 for small datasets, 10-50 for large datasets

3. **Noise Level**:
   - Higher noise: Use larger min_samples for robustness
   - Clean data: Can use smaller values

### Comparison with Other Clustering Algorithms

| Algorithm | Number of Clusters | Cluster Shape | Handles Noise | Scalability |
|-----------|-------------------|---------------|---------------|-------------|
| DBSCAN | Auto | Arbitrary | Yes | O(n log n) with index |
| K-Means | Required | Spherical | No | O(nk) |
| Hierarchical | Auto/Required | Arbitrary | No | O(n^2) or O(n^3) |
| GMM | Required | Elliptical | No | O(nk) |
| OPTICS | Auto | Arbitrary | Yes | O(n log n) |
| HDBSCAN | Auto | Arbitrary | Yes | O(n log n) |

In [ ]:
# Demonstrate DBSCAN limitation with varying densities

# Create dataset with varying density clusters
np.random.seed(42)
# Dense cluster
cluster1 = np.random.randn(200, 2) * 0.3 + np.array([0, 0])
# Sparse cluster
cluster2 = np.random.randn(200, 2) * 1.5 + np.array([5, 5])
X_varying = np.vstack([cluster1, cluster2])

# Try different eps values
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

eps_values = [0.3, 0.7, 1.5]
for idx, eps in enumerate(eps_values):
    dbscan = DBSCANScratch(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_varying)
    
    unique_labels = set(labels)
    colors = [plt.cm.viridis(l / max(1, max(unique_labels) if max(unique_labels) > 0 else 1)) 
              if l != -1 else 'red' for l in labels]
    
    axes[idx].scatter(X_varying[:, 0], X_varying[:, 1], c=colors, s=20, alpha=0.7)
    n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)
    axes[idx].set_title(f'eps={eps}\nClusters: {n_clusters}, Noise: {n_noise}')
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.suptitle('DBSCAN Limitation: Varying Density Clusters\n'
            '(Left: Only dense cluster found, Right: Clusters merged)', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

print("Note: No single eps value can correctly identify both clusters.")
print("For varying density data, consider using OPTICS or HDBSCAN instead.")

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn
import time

def compare_implementations(X, eps, min_samples, dataset_name):
    """
    Compare our DBSCAN implementation with sklearn's DBSCAN.
    
    Parameters:
    -----------
    X : ndarray
        Input data.
    eps : float
        Epsilon parameter.
    min_samples : int
        Minimum samples parameter.
    dataset_name : str
        Name of the dataset for display.
    """
    print(f"\n{'='*60}")
    print(f"Comparison on {dataset_name}")
    print(f"Parameters: eps={eps}, min_samples={min_samples}")
    print(f"{'='*60}")
    
    # Our implementation
    start_time = time.time()
    our_dbscan = DBSCANScratch(eps=eps, min_samples=min_samples)
    our_labels = our_dbscan.fit_predict(X)
    our_time = time.time() - start_time
    
    # Sklearn implementation
    start_time = time.time()
    sklearn_dbscan = SklearnDBSCAN(eps=eps, min_samples=min_samples)
    sklearn_labels = sklearn_dbscan.fit_predict(X)
    sklearn_time = time.time() - start_time
    
    # Results comparison
    print(f"\nOur Implementation:")
    print(f"  Time: {our_time:.4f} seconds")
    print(f"  Clusters found: {our_dbscan.n_clusters_}")
    print(f"  Noise points: {our_dbscan.n_noise_}")
    print(f"  Core points: {len(our_dbscan.core_sample_indices_)}")
    
    sklearn_n_clusters = len(set(sklearn_labels)) - (1 if -1 in sklearn_labels else 0)
    sklearn_n_noise = np.sum(sklearn_labels == -1)
    
    print(f"\nSklearn Implementation:")
    print(f"  Time: {sklearn_time:.4f} seconds")
    print(f"  Clusters found: {sklearn_n_clusters}")
    print(f"  Noise points: {sklearn_n_noise}")
    print(f"  Core points: {len(sklearn_dbscan.core_sample_indices_)}")
    
    # Check label agreement (labels may differ in numbering)
    # Use Adjusted Rand Index for comparison
    ari = adjusted_rand_score(our_labels, sklearn_labels)
    print(f"\nAdjusted Rand Index between implementations: {ari:.4f}")
    
    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Our implementation
    unique_labels = set(our_labels)
    colors_our = [plt.cm.viridis(l / max(1, max(unique_labels) if max(unique_labels) > 0 else 1)) 
                  if l != -1 else 'red' for l in our_labels]
    axes[0].scatter(X[:, 0], X[:, 1], c=colors_our, s=20, alpha=0.7)
    axes[0].set_title(f'Our Implementation\nClusters: {our_dbscan.n_clusters_}, '
                     f'Noise: {our_dbscan.n_noise_}')
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    
    # Sklearn implementation
    unique_labels_sk = set(sklearn_labels)
    colors_sk = [plt.cm.viridis(l / max(1, max(unique_labels_sk) if max(unique_labels_sk) > 0 else 1)) 
                 if l != -1 else 'red' for l in sklearn_labels]
    axes[1].scatter(X[:, 0], X[:, 1], c=colors_sk, s=20, alpha=0.7)
    axes[1].set_title(f'Sklearn Implementation\nClusters: {sklearn_n_clusters}, '
                     f'Noise: {sklearn_n_noise}')
    axes[1].set_xlabel('Feature 1')
    axes[1].set_ylabel('Feature 2')
    
    plt.suptitle(f'{dataset_name} - Implementation Comparison', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return our_labels, sklearn_labels

# Compare on moons dataset
our_moons, sk_moons = compare_implementations(X_moons_scaled, eps=0.3, min_samples=5, 
                                              dataset_name="Moons Dataset")

In [ ]:
# Compare on circles dataset
our_circles, sk_circles = compare_implementations(X_circles_scaled, eps=0.2, min_samples=5, 
                                                  dataset_name="Circles Dataset")

In [ ]:
# Compare on custom dataset
our_custom, sk_custom = compare_implementations(X_custom_scaled, eps=0.4, min_samples=5, 
                                                dataset_name="Custom Dataset")

In [ ]:
# Performance comparison with different dataset sizes
def benchmark_performance(sizes=[100, 200, 500, 1000], eps=0.3, min_samples=5):
    """
    Benchmark performance of both implementations across different dataset sizes.
    """
    our_times = []
    sklearn_times = []
    
    for size in sizes:
        X, _ = make_moons(n_samples=size, noise=0.1, random_state=42)
        X = StandardScaler().fit_transform(X)
        
        # Our implementation
        start = time.time()
        our_dbscan = DBSCANScratch(eps=eps, min_samples=min_samples)
        _ = our_dbscan.fit_predict(X)
        our_times.append(time.time() - start)
        
        # Sklearn implementation
        start = time.time()
        sklearn_dbscan = SklearnDBSCAN(eps=eps, min_samples=min_samples)
        _ = sklearn_dbscan.fit_predict(X)
        sklearn_times.append(time.time() - start)
    
    # Plot results
    plt.figure(figsize=(10, 5))
    plt.plot(sizes, our_times, 'o-', label='Our Implementation', linewidth=2, markersize=8)
    plt.plot(sizes, sklearn_times, 's-', label='Sklearn', linewidth=2, markersize=8)
    plt.xlabel('Dataset Size (n_samples)')
    plt.ylabel('Time (seconds)')
    plt.title('Performance Comparison: Our Implementation vs Sklearn')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\nPerformance Summary:")
    print("-" * 50)
    print(f"{'Size':<10} {'Our Time (s)':<15} {'Sklearn Time (s)':<15} {'Ratio':<10}")
    print("-" * 50)
    for i, size in enumerate(sizes):
        ratio = our_times[i] / sklearn_times[i] if sklearn_times[i] > 0 else float('inf')
        print(f"{size:<10} {our_times[i]:<15.4f} {sklearn_times[i]:<15.4f} {ratio:<10.2f}x")

# Run benchmark
benchmark_performance(sizes=[100, 200, 500, 1000])

## Summary & Key Takeaways

### What We Learned:

1. **Core Concepts**: DBSCAN identifies clusters based on density, classifying points as core, border, or noise.

2. **Implementation Details**: 
   - Precompute distance matrix for efficiency (trading memory for speed)
   - Use vectorized operations where possible
   - BFS-style cluster expansion

3. **Parameter Selection**:
   - Use k-distance plot to find optimal eps
   - min_samples should be at least D + 1 (dimensionality + 1)

4. **Strengths**:
   - Discovers arbitrary-shaped clusters
   - Automatically identifies noise/outliers
   - Does not require specifying number of clusters

5. **Limitations**:
   - Struggles with varying density clusters
   - Sensitive to parameter choices
   - Performance degrades in high dimensions

### Practical Tips:

1. **Always scale your data** before applying DBSCAN
2. **Use the k-distance plot** to guide eps selection
3. **Start with min_samples = 2 * n_features** as a baseline
4. **Evaluate using silhouette score** (excluding noise points)
5. **For varying densities**, consider OPTICS or HDBSCAN
6. **For large datasets**, use sklearn with optimized spatial indexing

### Next Steps:

- Explore OPTICS for hierarchical density-based clustering
- Try HDBSCAN for automatic eps selection
- Implement spatial indexing (KD-Tree) for improved performance
- Combine with dimensionality reduction for high-dimensional data